# 🌳 MitiWatch: Forest Loss Detection (Mabira, Uganda)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/mitiwatch/blob/main/notebooks/01_mitiwatch_detection.ipynb)

**Problem:** forest clearing in Uganda's reserves is usually discovered months late.
**MVP:** compare two dry-season Sentinel-2 images, find pixels that were forest and lost their greenness, and map them as clearing patches.

Run the cells from top to bottom. You need a free Google Earth Engine account linked to a Google Cloud project.

In [ ]:
# Colab already has these; this makes sure they are up to date
!pip install -q -U earthengine-api geemap

## 1. Connect to Earth Engine
Replace `your-project-id` with your Google Cloud project id (the one registered for Earth Engine).

In [ ]:
import ee, geemap

EE_PROJECT = "your-project-id"   # <-- change this

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

## 2. Settings

In [ ]:
CONFIG = dict(
    reserve_name="Mabira",
    fallback_bbox=[32.85, 0.33, 33.08, 0.52],   # used if the reserve is not in WDPA
    before=("2024-12-01", "2025-02-28"),        # dry season, year 1
    after=("2025-12-01", "2026-02-28"),         # dry season, year 2
    cloud_score_min=0.60,
    forest_ndvi_min=0.60,
    ndvi_drop_max=-0.25,
    after_ndvi_max=0.45,
    min_patch_pixels=25,                        # 0.25 ha
    scale=10,
)

## 3. Area of interest (forest reserve boundary)

In [ ]:
match = (ee.FeatureCollection("WCMC/WDPA/current/polygons")
         .filter(ee.Filter.eq("ISO3", "UGA"))
         .filter(ee.Filter.stringContains("NAME", CONFIG["reserve_name"])))
print("WDPA matches:", match.size().getInfo(), "(0 = using fallback box)")
aoi = ee.Geometry(ee.Algorithms.If(match.size().gt(0), match.geometry(),
                                   ee.Geometry.Rectangle(CONFIG["fallback_bbox"])))

## 4. Cloud-free composites + NDVI
Sentinel-2 images are masked with **Cloud Score+**, then combined with a median.

In [ ]:
BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]
cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

def clean_composite(start, end):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(aoi).filterDate(start, end)
           .linkCollection(cs_plus, ["cs_cdf"])
           .map(lambda img: img.select(BANDS).divide(10000)
                .updateMask(img.select("cs_cdf").gte(CONFIG["cloud_score_min"]))))
    print(f"{start} to {end}: {col.size().getInfo()} images")
    img = col.median().clip(aoi)
    return img.addBands(img.normalizedDifference(["B8", "B4"]).rename("NDVI"))

before = clean_composite(*CONFIG["before"])
after = clean_composite(*CONFIG["after"])

## 5. Detect new forest loss

In [ ]:
ndvi_b, ndvi_a = before.select("NDVI"), after.select("NDVI")
d_ndvi = ndvi_a.subtract(ndvi_b).rename("dNDVI")
forest_before = ndvi_b.gte(CONFIG["forest_ndvi_min"])

raw = (forest_before.And(d_ndvi.lte(CONFIG["ndvi_drop_max"]))
       .And(ndvi_a.lt(CONFIG["after_ndvi_max"])).selfMask())
loss = raw.updateMask(raw.connectedPixelCount(256, True)
                      .gte(CONFIG["min_patch_pixels"])).rename("loss")

def area_ha(mask):
    s = mask.multiply(ee.Image.pixelArea()).divide(10000).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=aoi, scale=CONFIG["scale"],
        maxPixels=1e10, bestEffort=True)
    return ee.Number(s.values().get(0))

forest_ha = area_ha(forest_before.selfMask()).getInfo()
loss_ha = area_ha(loss).getInfo()
print(f"Forest before: {forest_ha:,.0f} ha")
print(f"New loss:      {loss_ha:,.1f} ha ({100 * loss_ha / forest_ha:.2f}%)")

## 6. Before / after map
Drag the slider. Left = before, right = after with **red = new clearing**.

In [ ]:
rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.15, "gamma": 1.2}
m = geemap.Map(center=[0.43, 32.96], zoom=12)
left = geemap.ee_tile_layer(before, rgb, "Before")
after_with_loss = after.visualize(**rgb).blend(loss.visualize(palette=["ff0000"]))
right = geemap.ee_tile_layer(after_with_loss, {}, "After + new loss")
m.split_map(left_layer=left, right_layer=right)
m.addLayer(loss, {"palette": ["ff0000"]}, "New forest loss")
m.addLayer(ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(aoi)]), 1, 2),
           {"palette": ["ffff00"]}, "Reserve boundary")
m

## 7. Loss patches (polygons with area)

In [ ]:
def describe(f):
    c = f.geometry().centroid(1).coordinates()
    return f.set({"area_ha": f.geometry().area(1).divide(10000),
                  "lon": c.get(0), "lat": c.get(1)})

patches = (loss.toInt().reduceToVectors(
              geometry=aoi, scale=CONFIG["scale"], geometryType="polygon",
              eightConnected=True, labelProperty="loss", maxPixels=1e10, bestEffort=True)
           .map(describe).sort("area_ha", False).limit(5000))

gdf = geemap.ee_to_gdf(patches)
print(len(gdf), "patches")
gdf[["area_ha", "lat", "lon"]].head(10)

## 8. Save results (for the Flask dashboard)
Downloads `patches.geojson`. Put it in the repo's `data/` folder.

In [ ]:
gdf.to_file("patches.geojson", driver="GeoJSON")
try:
    from google.colab import files
    files.download("patches.geojson")
except ImportError:
    print("Saved patches.geojson")

## Validate
Compare the red patches with **Global Forest Watch** (globalforestwatch.org) or the Hansen loss layer in the Code Editor script. If they line up, the detection works.

**Next (after the MVP):** AI false-alarm filter, SMS alerts to rangers, more forest reserves.